In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os

In [3]:
# -------------------------------------------------
# Device configuration
# -------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
class CrossAttention(nn.Module):
    #Attention is split into 8 parallel heads
    def __init__(self, dim=768, num_heads=8, dropout=0.3):#Each head learns different alignment patterns.
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True #[Batch, Sequence, Features]
        )
        self.norm = nn.LayerNorm(dim) #Normalizes across features, Prevents one modality from dominating

    def forward(self, query, key_value):
        """
        This function lets one token (query) attend to a sequence of tokens (key/value) and returns a context-aware enriched query.

        query: [B, 1, D]
        B → batch size
        1 → exactly one token
        D → embedding dimension (e.g., 768)

        key_value: [B, N, D]
        N → number of tokens in another modality
        Audio frames
        Video patches
        Text tokens

        """
        #cross-attention computation
        attn_out, _ = self.attn(query, key_value, key_value) #attn_out=αV , _ contains attention maps
       # Residual connection , LayerNorm —> stabilizing the fusion
        return self.norm(attn_out + query)


class MultimodalEmotionModel(nn.Module):
    def __init__(self, embed_dim=768, num_heads=8, num_classes=8, dropout=0.3):
        super().__init__()
        #Learned NULL embeddings
        """ These are learnable placeholders.
            If one modality is missing, the model uses a learned null vector instead of undefined prediction/behaviour.

            Importance:
            1.Handles missing audio / text / video
            2.NULL vectors are trained, not fixed zeros
            3.Model learns how “absence” should influence emotion

            Shape: [768]
        """
        self.null_text   = nn.Parameter(torch.zeros(embed_dim))
        self.null_vision = nn.Parameter(torch.zeros(embed_dim))
        self.null_audio  = nn.Parameter(torch.zeros(embed_dim))


        #Cross-attention layers
        #This creates bidirectional multimodal understanding.
        self.audio_to_tv = CrossAttention(embed_dim, num_heads)  #Audio attends to Text + Vision
        self.text_to_av  = CrossAttention(embed_dim, num_heads)  #Text attends to Audio + Vision
        self.vision_to_at = CrossAttention(embed_dim, num_heads) #Vision attends to Audio + Text

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 3, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, audio, text, vision, masks):

        # -------- MASKS --------
        text_mask   = masks[:, 0].unsqueeze(1)   # [B,1]
        vision_mask = masks[:, 1].unsqueeze(1)   # [B,1]
        audio_mask = masks[:, 2].unsqueeze(1)   # [B,1]

        # Replace missing modalities with learned NULL embeddings
        text = text_mask * text + (1 - text_mask) * self.null_text.unsqueeze(0)
        vision = vision_mask * vision + (1 - vision_mask) * self.null_vision.unsqueeze(0)
        audio = audio_mask * audio + (1 - audio_mask) * self.null_audio.unsqueeze(0)

        # -------- Attention Preparation --------
         #unsqueeze(1) turns a vector into a one-step sequence so attention can read it. ( attention only works on sequences)
        # [B,D](Batch, Features)->[B,T,D]([Batch, Sequence, Features])
        a = audio.unsqueeze(1)   # [B,1,D]
        t = text.unsqueeze(1)
        v = vision.unsqueeze(1)

        # Context pairs
        """
            Concatenation creates a context window of other modalities.
            a : [B, 1, 768]  → audio
            t : [B, 1, 768]  → text
            v : [B, 1, 768]  → vision

            Audio  → looks at [Text, Vision]
            Text   → looks at [Audio, Vision]
            Vision → looks at [Audio, Text]
        """

        tv = torch.cat([t, v], dim=1) #When AUDIO is thinking, let it look at both TEXT and VISION together.
        av = torch.cat([a, v], dim=1) #When TEXT is thinking, let it look at AUDIO and VISION together.
        at = torch.cat([a, t], dim=1) #When VISION is thinking, let it look at AUDIO and TEXT together.

        # Cross-attention fusion
        """
        Audio listens to text and vision, learns what matters, and becomes smarter

        a is the query (audio)
        tv is key/value (text + vision)
        Cross-attention happens inside

        a_fused = audio embedding enriched by text + vision

        """
        a_fused = self.audio_to_tv(a, tv).squeeze(1)
        t_fused = self.text_to_av(t, av).squeeze(1)
        v_fused = self.vision_to_at(v, at).squeeze(1)

        # Final fused representation
        fused = torch.cat([a_fused, t_fused, v_fused], dim=1)

        return self.classifier(fused)

#Load Audio extraction module

In [5]:
import os
import torch
import numpy as np
import soundfile as sf

from moviepy.editor import VideoFileClip
from transformers import Wav2Vec2Processor, Wav2Vec2Model

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



In [6]:
# -------------------------------------------------
# Load pretrained Wav2Vec2 model
# -------------------------------------------------
AUDIO_MODEL_NAME = "facebook/wav2vec2-base"
processor_audio = Wav2Vec2Processor.from_pretrained(AUDIO_MODEL_NAME)
model_audio = Wav2Vec2Model.from_pretrained(AUDIO_MODEL_NAME).to(DEVICE)
model_audio.eval()

The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Wav2Vec2Model(
  (feature_extractor): Wav2Vec2FeatureEncoder(
    (conv_layers): ModuleList(
      (0): Wav2Vec2GroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): Wav2Vec2FeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): Wav2Vec2Encoder(
    (pos_conv_embed): Wav2Vec2PositionalConvEmbedding(
  

In [7]:
# -------------------------------------------------
# Extract audio from MP4 using MoviePy
# -------------------------------------------------
def extract_audio_mp4_to_wav(mp4_path, wav_path):
    """
    Extracts audio from an MP4 file and saves it as a 16kHz mono WAV file.

    """
    video = VideoFileClip(mp4_path)

    if video.audio is None:
        video.close()
        raise ValueError(f"No audio stream found in {mp4_path}")

    video.audio.write_audiofile(
        wav_path,
        fps=16000,
        nbytes=2,
        codec="pcm_s16le",
        logger=None
    )

    video.close()

    # -------------------------------------------------
# Audio embedding extraction + saving
# -------------------------------------------------
@torch.no_grad()
def extract_audio_embedding(
    video_path,
    temp_wav_dir="_temp_wav"
):
    os.makedirs(temp_wav_dir, exist_ok=True)

    video_name = os.path.splitext(os.path.basename(video_path))[0]
    temp_wav_path = os.path.join(temp_wav_dir, f"{video_name}.wav")

    # 1. Extract WAV from MP4
    extract_audio_mp4_to_wav(video_path, temp_wav_path)

    # 2. Load WAV using soundfile
    waveform, sr = sf.read(temp_wav_path, dtype="float32")

    # Convert to mono if stereo
    if waveform.ndim == 2:
        waveform = waveform.mean(axis=1)

    waveform = torch.from_numpy(waveform)

    # 3. Prepare input for Wav2Vec2
    inputs = processor_audio(
        waveform,
        sampling_rate=sr,
        return_tensors="pt"
    ).to(DEVICE)

    # 4. Forward pass through Wav2Vec2
    outputs = model_audio(**inputs)

    # Frame-level embeddings: [time_steps, hidden_dim]
    hidden_states = outputs.last_hidden_state.squeeze(0)

    # -------------------------------------------------
    # ONE audio embedding per video (temporal mean)
    # -------------------------------------------------
    audio_embedding = hidden_states.mean(dim=0)  # [768]

    # L2 normalization
    audio_embedding = audio_embedding / audio_embedding.norm()

    # Cleanup temp WAV
    if os.path.exists(temp_wav_path):
        os.remove(temp_wav_path)

    return audio_embedding.cpu()


In [8]:
import os
import cv2
import torch
import numpy as np
from transformers import CLIPModel, CLIPProcessor

In [9]:
# -------------------------------------------------
# Load CLIP Vision model
# -------------------------------------------------
CLIP_NAME = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(CLIP_NAME)
model_visual = CLIPModel.from_pretrained(CLIP_NAME).to(DEVICE)
model_visual.eval()

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [10]:
# -------------------------------------------------
# Frame sampling using OpenCV
# -------------------------------------------------

def sample_frames_cv2(video_path, num_frames=8):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count == 0:
        cap.release()
        raise ValueError(f"No frames found in video: {video_path}")

    indices = np.linspace(0, frame_count - 1, num_frames, dtype=int)
    frames = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame_bgr = cap.read()

        if not ok:
            frame_rgb = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            frame_rgb = cv2.resize(frame_rgb, (224, 224), interpolation=cv2.INTER_AREA)

        frames.append(frame_rgb)

    cap.release()
    return frames

   # -------------------------------------------------
# CLIP visual feature extraction + saving
# -------------------------------------------------

@torch.no_grad()
def extract_visual_embedding(
    video_path,
    num_frames=8
):
    # Sample frames
    frames = sample_frames_cv2(video_path, num_frames)

    # Preprocess frames
    inputs = processor(images=frames, return_tensors="pt").to(DEVICE)

    # Forward pass through CLIP vision encoder
    vision_outputs = model_visual.vision_model(**inputs)

    # CLS token embeddings (frame-level)
    z_v = vision_outputs.last_hidden_state[:, 0, :]  # [num_frames, 768]

    # ---------------------------------------------
    # Video-level embedding (ONE vector per video)
    # ---------------------------------------------
    video_embedding = z_v.mean(dim=0)  # [768]

    #l2 normalization

    video_embedding = video_embedding / video_embedding.norm()

    return video_embedding.cpu()



#initialize text feature extraction module

In [11]:
!pip install openai-whisper


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 50.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=32e7a63df9c55795f9723f89b439150412851799334434a594b690b080ea9c25
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [12]:
#!pip install openai-whisper

import os
import torch
import numpy as np
import soundfile as sf
from moviepy.editor import VideoFileClip
import whisper
from transformers import AutoTokenizer, AutoModel

In [13]:
# -------------------------------------------------
# Load Whisper (Speech → Text)
# -------------------------------------------------
WHISPER_MODEL_SIZE = "base"  # tiny | base | small | medium
whisper_model = whisper.load_model(WHISPER_MODEL_SIZE, device=DEVICE)

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 469MiB/s]


In [14]:
# -------------------------------------------------
# Load Text Embedding Model
# -------------------------------------------------
TEXT_MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_model = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE)
text_model.eval()


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [15]:
# -------------------------------------------------
# Extract audio from MP4 using MoviePy
# -------------------------------------------------
def extract_audio_mp4_to_wav(mp4_path, wav_path):
    video = VideoFileClip(mp4_path)

    if video.audio is None:
        video.close()
        raise ValueError(f"No audio stream found in {mp4_path}")

    video.audio.write_audiofile(
        wav_path,
        fps=16000,
        nbytes=2,
        codec="pcm_s16le",
        logger=None
    )

    video.close()

    # -------------------------------------------------
# Text embedding extraction
# -------------------------------------------------
@torch.no_grad()
def extract_text_embedding(
    video_path,
    temp_wav_dir="_temp_wav"
):
    os.makedirs(temp_wav_dir, exist_ok=True)

    video_name = os.path.splitext(os.path.basename(video_path))[0]
    temp_wav_path = os.path.join(temp_wav_dir, f"{video_name}.wav")

    # 1. Extract WAV from MP4
    extract_audio_mp4_to_wav(video_path, temp_wav_path)

    # 2. Load audio
    audio, sr = sf.read(temp_wav_path, dtype="float32")

    if audio.ndim == 2:
        audio = audio.mean(axis=1)

    # Whisper expects 16 kHz
    if sr != 16000:
        audio = whisper.audio.resample(audio, sr, 16000)

    # 3. Speech → Text
    result = whisper_model.transcribe(audio, fp16=False)
    transcript = result["text"].strip()

    if not transcript:
        print(f"Warning: No transcript generated for {video_path}")
        return None

    # 4. Text → Embedding
    inputs = tokenizer(
        transcript,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(DEVICE)

    outputs = text_model(**inputs)

    # CLS token embedding
    text_embedding = outputs.last_hidden_state[:, 0, :].squeeze(0)

    # L2 normalization
    text_embedding = text_embedding / text_embedding.norm(p=2)

    # Cleanup
    if os.path.exists(temp_wav_path):
        os.remove(temp_wav_path)

    return text_embedding.cpu()

In [16]:
MODEL_LOAD_PATH = "/content/drive/MyDrive/Dissertion/model/multimodal_emotion_model.pth"


In [17]:

# ===============================
# Load Emotion Model
# ===============================

print("Loading Multimodal Model...")
model_em = MultimodalEmotionModel()
model_em.load_state_dict(torch.load(MODEL_LOAD_PATH, map_location=DEVICE))
model_em.to(DEVICE)
model_em.eval()

Loading Multimodal Model...


MultimodalEmotionModel(
  (audio_to_tv): CrossAttention(
    (attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
    )
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (text_to_av): CrossAttention(
    (attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
    )
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (vision_to_at): CrossAttention(
    (attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
    )
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (classifier): Sequential(
    (0): Linear(in_features=2304, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=8, bias=True)
  )
)

In [18]:
# Dataset ground truth (RAVDESS)
EMOTION_MAP_GT = {
    1: "Neutral",
    2: "Calm",
    3: "Happy",
    4: "Sad",
    5: "Angry",
    6: "Fearful",
    7: "Disgust",
    8: "Surprised",
}

# Model output indices (0-based)
EMOTION_MAP_MODEL = {
    0: "Neutral",
    1: "Calm",
    2: "Happy",
    3: "Sad",
    4: "Angry",
    5: "Fearful",
    6: "Disgust",
    7: "Surprised",
}


In [19]:
import os

def get_ground_truth(filename: str):
    try:
        code = int(filename.split("-")[2])
        return EMOTION_MAP_GT.get(code, "Unknown")
    except Exception:
        return "Unknown"



In [20]:
import pandas as pd
from tqdm import tqdm

DATA_DIR = "/content/drive/MyDrive/Dissertion/inference_test_data"

results = []

video_files = []

for root, _, files in os.walk(DATA_DIR):
    for f in files:
        if f.lower().endswith(".mp4"):
            video_files.append(os.path.join(root, f))

print("Total test videos:", len(video_files))



Total test videos: 88


In [21]:
# ===============================
#  Main Prediction Function
# ===============================

def predict_emotion(video_file):

    if video_file is None:
        return "❌ No video uploaded!"

    video_path = video_file

    print("🎥 Video received:", video_path)

        # -------- AUDIO --------
    a = extract_audio_embedding(video_path)
    audio_mask = 0 if a is None else 1
    a = (torch.zeros(768) if a is None else a).to(DEVICE)

    # -------- VISION --------
    v = extract_visual_embedding(video_path)
    vision_mask = 0 if v is None else 1
    v = (torch.zeros(768) if v is None else v).to(DEVICE)

    # -------- TEXT --------
    t = extract_text_embedding(video_path)
    text_mask = 0 if v is None else 1
    t = (torch.zeros(768) if t is None else t).to(DEVICE)

    # -------- MASKS --------
    masks = torch.tensor([[text_mask, vision_mask, audio_mask]], device=DEVICE)

    # -------- MODEL --------
    with torch.no_grad():
        logits = model_em(
            a.unsqueeze(0),
            t.unsqueeze(0),
            v.unsqueeze(0),
            masks
        )

        pred_class = torch.argmax(logits, dim=1).item()
        pred = EMOTION_MAP_MODEL[pred_class]

        return pred


In [22]:
for video_path in tqdm(video_files):

    filename = os.path.basename(video_path)

    # --- Ground truth ---
    gt = get_ground_truth(filename)

    # --- Prediction ---
    pred = predict_emotion(video_path)

    # --- Correct / Incorrect ---
    correct = pred.lower() == gt.lower()

    results.append({
        "Video": filename,
        "Ground Truth": gt,
        "Prediction": pred,
        "Correct": correct
    })




  0%|          | 0/88 [00:00<?, ?it/s]

🎥 Video received: /content/drive/MyDrive/Dissertion/inference_test_data/01-02-02-01-01-01-08.mp4


  1%|          | 1/88 [00:04<06:03,  4.18s/it]

🎥 Video received: /content/drive/MyDrive/Dissertion/inference_test_data/01-02-01-01-01-02-08.mp4


  2%|▏         | 2/88 [00:05<03:52,  2.70s/it]

...

 98%|█████████▊| 86/88 [03:08<00:04,  2.35s/it]

🎥 Video received: /content/drive/MyDrive/Dissertion/inference_test_data/02-02-06-01-02-02-08.mp4


 99%|█████████▉| 87/88 [03:10<00:02,  2.30s/it]

🎥 Video received: /content/drive/MyDrive/Dissertion/inference_test_data/02-02-06-01-01-01-08.mp4


100%|██████████| 88/88 [03:13<00:00,  2.20s/it]

In [23]:
df = pd.DataFrame(results)

print("\n=== Inference Test Report ===")
display(df)



=== Inference Test Report ===


,Video,Ground Truth,Prediction,Correct
0,01-02-02-01-01-01-08.mp4,Calm,Calm,True
1,01-02-01-01-01-02-08.mp4,Neutral,Neutral,True
2,01-02-01-01-02-02-08.mp4,Neutral,Calm,False
3,01-02-02-01-02-01-08.mp4,Calm,Happy,False
4,01-02-01-01-02-01-08.mp4,Neutral,Calm,False
...,...,...,...,...
83,02-02-06-01-02-01-08.mp4,Fearful,Fearful,True
84,02-02-06-01-01-02-08.mp4,Fearful,Fearful,True
85,02-02-06-02-01-02-08.mp4,Fearful,Sad,False
86,02-02-06-01-02-02-08.mp4,Fearful,Fearful,True


In [25]:
accuracy = df["Correct"].mean()

print("\n=== Statistics ===")
print(f"Total Samples : {len(df)}")
print(f"Correct       : {df['Correct'].sum()}")
print(f"Accuracy      : {accuracy:.4f}")



=== Statistics ===
Total Samples : 88
Correct       : 63
Accuracy      : 0.7159
